In [1]:
navy_urls = {
    "Total_Naval_Assets": "https://www.globalfirepower.com/navy-ships.php",
    "Naval_Tonnage": "https://www.globalfirepower.com/navy-force-by-tonnage.php",
    "Aircraft_Carriers": "https://www.globalfirepower.com/navy-aircraft-carriers.php",
    "Helicopter_Carriers": "https://www.globalfirepower.com/navy-helo-carriers.php",
    "Submarines": "https://www.globalfirepower.com/navy-submarines.php",
    "Destroyers": "https://www.globalfirepower.com/navy-destroyers.php",
    "Frigates": "https://www.globalfirepower.com/navy-frigates.php",
    "Corvettes": "https://www.globalfirepower.com/navy-corvettes.php",
    "Patrol_Vessels": "https://www.globalfirepower.com/navy-patrol-coastal-craft.php",
    "Mine_Warfare_Vessels": "https://www.globalfirepower.com/navy-mine-warfare-craft.php"
}

In [2]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [4]:
def scrape_metric(url, metric_name):

    print(f"Scraping {metric_name}...")

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed to load {url}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    records = soup.find_all("div", class_="recordsetContainer")

    data = []

    for record in records:

        long_name = record.find("div", class_="longFormName")
        short_name = record.find("div", class_="shortFormName")
        value = record.find("div", class_="valueContainer")

        if value:

            if long_name:
                country = long_name.get_text(strip=True)

            elif short_name:
                country = short_name.get_text(strip=True)

            else:
                continue

            value = value.get_text(strip=True)

            data.append([country, value])

    df = pd.DataFrame(data, columns=["Country", metric_name])

    return df

In [5]:
dfs = []

for metric, url in navy_urls.items():

    df = scrape_metric(url, metric)

    if df is not None:
        dfs.append(df)

    time.sleep(2)

Scraping Total_Naval_Assets...
Scraping Naval_Tonnage...
Scraping Aircraft_Carriers...
Scraping Helicopter_Carriers...
Scraping Submarines...
Scraping Destroyers...
Scraping Frigates...
Scraping Corvettes...
Scraping Patrol_Vessels...
Scraping Mine_Warfare_Vessels...


In [6]:
navy_df = dfs[0]

for df in dfs[1:]:

    navy_df = navy_df.merge(
        df,
        on="Country",
        how="outer"
    )

In [7]:
for col in navy_df.columns[1:]:

    navy_df[col] = (
        navy_df[col]
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
    )

    navy_df[col] = pd.to_numeric(
        navy_df[col],
        errors="coerce"
    )

In [8]:
print(navy_df.head())

       Country  Total_Naval_Assets  Naval_Tonnage  Aircraft_Carriers  \
0  Afghanistan                   0            NaN                  0   
1      Albania                  33            NaN                  0   
2      Algeria                 111        83480.0                  0   
3       Angola                  32            NaN                  0   
4    Argentina                  42       122128.0                  0   

   Helicopter_Carriers  Submarines  Destroyers  Frigates  Corvettes  \
0                    0           0           0         0          0   
1                    0           0           0         0          0   
2                    0           6           0         8          8   
3                    0           0           0         0          0   
4                    0           2           3         0          6   

   Patrol_Vessels  Mine_Warfare_Vessels  
0               0                     0  
1              33                     0  
2             

In [9]:
navy_df.to_csv("navy.csv", index=False)

print("Navy dataset saved successfully!")

Navy dataset saved successfully!


In [10]:
print("Shape:", navy_df.shape)
print()

print(navy_df.info())
print()

print(navy_df.head(10))

Shape: (145, 11)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Country               145 non-null    object 
 1   Total_Naval_Assets    145 non-null    int64  
 2   Naval_Tonnage         51 non-null     float64
 3   Aircraft_Carriers     145 non-null    int64  
 4   Helicopter_Carriers   145 non-null    int64  
 5   Submarines            145 non-null    int64  
 6   Destroyers            145 non-null    int64  
 7   Frigates              145 non-null    int64  
 8   Corvettes             145 non-null    int64  
 9   Patrol_Vessels        145 non-null    int64  
 10  Mine_Warfare_Vessels  145 non-null    int64  
dtypes: float64(1), int64(9), object(1)
memory usage: 12.6+ KB
None

       Country  Total_Naval_Assets  Naval_Tonnage  Aircraft_Carriers  \
0  Afghanistan                   0            NaN                  0   
1